# Native SmolVLM-256M no-pruner finetune
Two Kaggle GPUs, FP32, per-rank batch 4, accumulation 8 (effective batch 64). The frozen native vision encoder produces 1024 x 768 raw features per camera; the pretrained connector produces 64 x 576 tokens. All 384 visual tokens enter the causal language model. Connector, language model and head train from pretrained weights; there is no alignment stage.

Run cells in order. Full raw feature cache is about 35.8 GiB for 4072 distinct camera groups, plus prepared data, checkpoints and a required 6 GiB free reserve. Required space is checked from remote file metadata before downloads. GPU fit, throughput and quality remain unmeasured. A valid cache plus matching prepared JSON does not require raw photos.


In [ ]:
import os, json, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
GRADIENT_CHECKPOINTING = True
if type(GRADIENT_CHECKPOINTING) is not bool:
    raise TypeError('GRADIENT_CHECKPOINTING must be a bool')
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/huyvanzzz/Group_HND_car.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'huy-smolvlm-256m-no-pruner')
PREPARED_DATA_HF_REPO_ID = ''  # Dataset repo: manifest.json, train.jsonl, val.jsonl, test.jsonl at root.
CACHE_HF_REPO_ID = ''  # Separate dataset repo: features.float16.memmap, index.json, manifest.json at root.
FINETUNE_CHECKPOINT_HF_REPO_ID = ''  # Optional model repo: checkpoint plus <filename>.rng/**.
FINETUNE_CHECKPOINT_FILENAME = 'finetune_latest.pt'
if PREPARED_DATA_HF_REPO_ID and PREPARED_DATA_HF_REPO_ID == CACHE_HF_REPO_ID:
    raise ValueError('Prepared data and raw vision cache need separate dataset repositories')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception as exc:
    print('HF_TOKEN was not loaded:', type(exc).__name__)
print({'branch': GITHUB_BRANCH, 'gradient_checkpointing': GRADIENT_CHECKPOINTING,
       'hf_token_present': bool(os.environ.get('HF_TOKEN'))})


In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/kaggle/working/Efficient_VLM_For_Autonomous_Driving')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', GITHUB_BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', GITHUB_BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', GITHUB_BRANCH], cwd=REPO, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers', '-e', '.',
                'transformers==4.46.3', 'tokenizers==0.20.4', 'accelerate==1.15.0', 'pycocoevalcap'], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.environ['PYTHONPATH'] = str(REPO / 'src')
import torch, transformers, tokenizers, accelerate
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
assert accelerate.__version__ == '1.15.0'
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Enable two Kaggle GPUs'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import yaml, shutil
from huggingface_hub import hf_hub_download, hf_hub_url, get_hf_file_metadata, snapshot_download
from efficient_vlm_ad.smolvlm.config import load_config
from efficient_vlm_ad.hf_data import validate_split_counts
TRACKED_CONFIG = Path('configs/smolvlm_256m_kaggle_2gpu_no_pruner.yaml')
RUNTIME_CONFIG = Path('/kaggle/working/smolvlm_no_pruner_runtime.yaml')
raw = yaml.safe_load(TRACKED_CONFIG.read_text(encoding='utf-8'))
if type(GRADIENT_CHECKPOINTING) is not bool:
    raise TypeError('GRADIENT_CHECKPOINTING must be a bool')
raw['training']['gradient_checkpointing'] = GRADIENT_CHECKPOINTING
RUNTIME_CONFIG.write_text(yaml.safe_dump(raw, sort_keys=False), encoding='utf-8')
cfg = load_config(RUNTIME_CONFIG)
assert cfg.runtime.precision == 'fp32'
PROFILE = Path(cfg.project.output_dir)
PROFILE.mkdir(parents=True, exist_ok=True)

def run_backend(command, *args):
    subprocess.run([sys.executable, '-m', 'efficient_vlm_ad.smolvlm', command,
                    '--config', str(RUNTIME_CONFIG), *args], check=True)

def download_exact(repo_id, repo_type, destination, filenames):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    sizes = []
    for filename in filenames:
        url = hf_hub_url(repo_id=repo_id, repo_type=repo_type, filename=filename)
        metadata = get_hf_file_metadata(url, token=os.environ.get('HF_TOKEN'))
        if type(metadata.size) is not int or metadata.size < 0:
            raise ValueError(f'Cannot establish remote file size: {filename}')
        sizes.append(metadata.size)
    required = sum(sizes) + 6 * 1024**3
    free = shutil.disk_usage(destination).free
    if free < required:
        raise RuntimeError(f'Insufficient disk space: need {required / 1024**3:.2f} GiB including 6 GiB reserve; free {free / 1024**3:.2f} GiB')
    print(f'Downloading {sum(sizes) / 1024**3:.2f} GiB directly to {destination}')
    for filename in filenames:
        hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=filename, local_dir=destination)

def validate_prepared(prepared):
    manifest = json.loads((prepared / 'manifest.json').read_text(encoding='utf-8'))
    if manifest.get('subset') != 'full':
        raise ValueError('Full training requires full prepared data')
    counts = {}
    for split in ('train', 'val', 'test'):
        with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
            counts[split] = sum(1 for line in stream if line.strip() and json.loads(line))
    if counts != manifest['counts']:
        raise ValueError('Prepared manifest counts disagree with JSONL files')
    validate_split_counts(counts, cfg.data.expected_counts, subset='full')
    print('Prepared counts:', counts)
print({'config': str(RUNTIME_CONFIG), 'output': str(PROFILE), 'effective_batch': 4 * 8 * 2})


In [ ]:
prepared = PROFILE / 'prepared_data'
cache = Path(cfg.cache.dir)
prepared_files = ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl')
cache_files = ('features.float16.memmap', 'index.json', 'manifest.json')
if not all((cache / name).is_file() for name in cache_files):
    if CACHE_HF_REPO_ID:
        download_exact(CACHE_HF_REPO_ID, 'dataset', cache, cache_files)
    else:
        print('Cache missing: materialize full images and rebuild prepared data before extraction.')
        # Full local extraction also needs room for raw photos; estimate cache after preparation.
        run_backend('prepare-data', '--subset', 'full')
        validate_prepared(prepared)
        unique_groups = set()
        for split in ('train', 'val', 'test'):
            with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
                for line in stream:
                    if line.strip():
                        paths = json.loads(line)['camera_paths']
                        unique_groups.add(tuple(paths[name] for name in cfg.data.view_order))
        required = len(unique_groups) * 6 * 1024 * 768 * 2 + 6 * 1024**3
        if shutil.disk_usage(PROFILE).free < required:
            raise RuntimeError('Insufficient disk space for native feature cache plus 6 GiB reserve')
        run_backend('prepare-features', '--subset', 'full')
if not all((prepared / name).is_file() for name in prepared_files):
    if PREPARED_DATA_HF_REPO_ID:
        download_exact(PREPARED_DATA_HF_REPO_ID, 'dataset', prepared, prepared_files)
    else:
        run_backend('prepare-data', '--subset', 'full')
validate_prepared(prepared)
# Backend validation checks native geometry, processor policy, identity, index and byte length.
# It intentionally does not require raw photos for an already valid feature cache.
run_backend('validate-cache')


In [ ]:
run_backend('diagnose-train', '--debug', '--debug-samples', '1', '--debug-numerics')


In [ ]:
# Single process: compare real full-window and tail gradients before training.
run_backend('verify-accumulation', '--debug', '--debug-numerics')


In [ ]:
checkpoint_dir = PROFILE / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
LOCAL_LATEST = checkpoint_dir / 'finetune_latest.pt'
configured_local = checkpoint_dir / FINETUNE_CHECKPOINT_FILENAME
resume_path = None
if LOCAL_LATEST.is_file():
    resume_path = LOCAL_LATEST
elif configured_local.is_file():
    resume_path = configured_local
elif FINETUNE_CHECKPOINT_HF_REPO_ID:
    snapshot_download(repo_id=FINETUNE_CHECKPOINT_HF_REPO_ID, repo_type='model', local_dir=checkpoint_dir,
                      allow_patterns=[FINETUNE_CHECKPOINT_FILENAME, f'{FINETUNE_CHECKPOINT_FILENAME}.rng/**'])
    resume_path = configured_local
resume_args = []
if resume_path is not None:
    for path in (resume_path, Path(str(resume_path) + '.rng/manifest.json'),
                 Path(str(resume_path) + '.rng/rank_00000.pt'), Path(str(resume_path) + '.rng/rank_00001.pt')):
        if not path.is_file():
            raise FileNotFoundError(path)
    manifest = json.loads(Path(str(resume_path) + '.rng/manifest.json').read_text(encoding='utf-8'))
    if manifest['world_size'] != 2:
        raise ValueError('Resume checkpoint must contain two ranks of RNG state')
    # Standalone validator checks stored world size, metadata and sidecars before launch.
    run_backend('verify-resume-checkpoint', '--checkpoint', str(resume_path), '--stage', 'finetune')
    resume_args = ['--resume', str(resume_path)]
else:
    print('Starting from pretrained native vision encoder, connector, causal LM and head.')
subprocess.run([sys.executable, '-m', 'accelerate.commands.launch', '--multi_gpu', '--num_processes', '2',
                '--num_machines', '1', '--mixed_precision', 'no', '--dynamo_backend', 'no',
                '-m', 'efficient_vlm_ad.smolvlm', 'train', '--config', str(RUNTIME_CONFIG),
                '--stage', 'finetune', *resume_args, '--debug', '--debug-samples', '1', '--debug-numerics'], check=True)


In [ ]:
checkpoint = PROFILE / 'checkpoints/finetune_best.pt'
if not checkpoint.is_file():
    checkpoint = PROFILE / 'checkpoints/finetune_latest.pt'
run_backend('evaluate', '--checkpoint', str(checkpoint), '--batch-size', '4')
run_backend('benchmark', '--checkpoint', str(checkpoint), '--max-samples', '200')
for filename in ('metrics.json', 'benchmark.json'):
    print(filename, (PROFILE / filename).read_text(encoding='utf-8'))
print(''.join((PROFILE / 'predictions.jsonl').read_text(encoding='utf-8').splitlines(keepends=True)[:8]))


In [ ]:
import tarfile
archive = Path('/kaggle/working/smolvlm_256m_no_pruner_finetune_artifacts.tar.gz')
# Explicit allowlist excludes the large cache and raw photos.
with tarfile.open(archive, 'w:gz') as bundle:
    bundle.add(RUNTIME_CONFIG, arcname=RUNTIME_CONFIG.name)
    for name in ('checkpoints', 'prepared_data', 'metrics.json', 'benchmark.json', 'predictions.jsonl', 'debug'):
        path = PROFILE / name
        if path.exists():
            bundle.add(path, arcname=f'{PROFILE.name}/{name}')
print(archive, archive.stat().st_size)
